In [ ]:
%pip install finance-datareader
from pathlib import Path

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import FinanceDataReader as fdr

print("pandas version:", pd.__version__)
print("FinanceDataReader version:", fdr.__version__)

pandas version: 3.0.3
FinanceDataReader version: 0.9.202


USD_KRW는 초기 수집·검토용이며, 최종 환율 분석에는 아래에서 직접 읽은 FRED CSV를 사용한다.

In [5]:
symbols = {
    "KB금융": "105560",
    "신한지주": "055550",
    "하나금융지주": "086790",
    "우리금융지주": "316140",
    "KOSPI": "KS11",
    "USD_KRW": "USD/KRW"
}

start_date = "2021-01-01"
end_date = "2025-12-31"

symbols

{'KB금융': '105560',
 '신한지주': '055550',
 '하나금융지주': '086790',
 '우리금융지주': '316140',
 'KOSPI': 'KS11',
 'USD_KRW': 'USD/KRW'}

In [7]:
market_data_raw = {}

for name, code in symbols.items():
    data = fdr.DataReader(code, start_date, end_date)
    market_data_raw[name] = data
    print(name, data.shape)

KB금융 (1225, 6)
신한지주 (1225, 6)
하나금융지주 (1225, 6)
우리금융지주 (1225, 6)
KOSPI (1225, 10)
USD_KRW (1304, 6)


In [9]:
raw_dir = Path.cwd().parent / "data" / "raw"

file_names = {
    "KB금융": "kb_financial.csv",
    "신한지주": "shinhan_financial.csv",
    "하나금융지주": "hana_financial.csv",
    "우리금융지주": "woori_financial.csv",
    "KOSPI": "kospi.csv",
    "USD_KRW": "usd_krw.csv"
}

for name, data in market_data_raw.items():
    file_path = raw_dir / file_names[name]
    data.to_csv(file_path, encoding="utf-8-sig")
    print(file_path.name, "저장 완료")

kb_financial.csv 저장 완료
shinhan_financial.csv 저장 완료
hana_financial.csv 저장 완료
woori_financial.csv 저장 완료
kospi.csv 저장 완료
usd_krw.csv 저장 완료


In [10]:
base_rate_raw = pd.DataFrame({
    "change_date": pd.to_datetime([
        "2020-05-28",
        "2021-08-26",
        "2021-11-25",
        "2022-01-14",
        "2022-04-14",
        "2022-05-26",
        "2022-07-13",
        "2022-08-25",
        "2022-10-12",
        "2022-11-24",
        "2023-01-13",
        "2024-10-11",
        "2024-11-28",
        "2025-02-25",
        "2025-05-29"
    ]),
    "base_rate": [
        0.50,
        0.75,
        1.00,
        1.25,
        1.50,
        1.75,
        2.25,
        2.50,
        3.00,
        3.25,
        3.50,
        3.25,
        3.00,
        2.75,
        2.50
    ]
})

base_rate_raw

,change_date,base_rate
0,2020-05-28,0.50
1,2021-08-26,0.75
2,2021-11-25,1.00
3,2022-01-14,1.25
4,2022-04-14,1.50
5,2022-05-26,1.75
6,2022-07-13,2.25
7,2022-08-25,2.50
8,2022-10-12,3.00
9,2022-11-24,3.25


In [11]:
base_rate_path = raw_dir / "bok_base_rate.csv"

base_rate_raw.to_csv(
    base_rate_path,
    index=False,
    encoding="utf-8-sig"
)

print(base_rate_path.name, "저장 완료")

bok_base_rate.csv 저장 완료


### 기준금리 데이터 출처

- 출처: 한국은행 「한국은행 기준금리 추이」
- URL: https://www.bok.or.kr/portal/singl/baseRate/list.do?menuNo=200643
- 수집 방법: 한국은행이 공표한 기준금리 변경일자와 금리를 분석 가능한 표 형태로 구조화
- 수집 범위: 2020-05-28 ~ 2025-05-29
- 참고: 2021-01-01 당시 적용 금리를 반영하기 위해 2020-05-28 자료를 포함

In [12]:
print("크기:", base_rate_raw.shape)
print("시작일:", base_rate_raw["change_date"].min())
print("종료일:", base_rate_raw["change_date"].max())
print("\n자료형:")
print(base_rate_raw.dtypes)

크기: (15, 2)
시작일: 2020-05-28 00:00:00
종료일: 2025-05-29 00:00:00

자료형:
change_date    datetime64[us]
base_rate             float64
dtype: object


FRED 원본 CSV를 직접 읽고, 결측 표시 "."을 NaN으로 처리해 유지한다.

In [19]:
fred_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    f"?id=DEXKOUS&cosd={start_date}&coed={end_date}"
)

In [20]:
fred_fx_raw = pd.read_csv(
    fred_url,
    index_col = 0,
    parse_dates = True,
    na_values="."
)

In [21]:
print(fred_fx_raw.loc["2025-12-24":"2025-12-26"])

                  DEXKOUS
observation_date         
2025-12-24        1443.62
2025-12-25            NaN
2025-12-26        1441.16


In [22]:
fred_fx_raw["DEXKOUS"].isna().sum()

np.int64(54)

In [ ]:
kb_path = raw_dir / "kb_financial.csv"
kb_dates = pd.to_datetime(pd.read_csv(kb_path)["Date"])

fx_on_kb_days_raw = fred_fx_raw.reindex(kb_dates)

In [26]:
print(fx_on_kb_days_raw.shape)
print(fx_on_kb_days_raw["DEXKOUS"].isna().sum())

(1225, 1)
44


In [28]:
missing_fx_days = fx_on_kb_days_raw[fx_on_kb_days_raw["DEXKOUS"].isna()]
print(missing_fx_days)

            DEXKOUS
Date               
2021-01-18      NaN
2021-01-20      NaN
2021-02-15      NaN
2021-05-31      NaN
2021-07-05      NaN
2021-09-06      NaN
2021-11-11      NaN
2021-11-25      NaN
2021-12-24      NaN
2022-01-17      NaN
2022-02-21      NaN
2022-05-30      NaN
2022-06-20      NaN
2022-07-04      NaN
2022-09-05      NaN
2022-11-11      NaN
2022-11-24      NaN
2022-12-26      NaN
2023-01-02      NaN
2023-01-16      NaN
2023-02-20      NaN
2023-06-19      NaN
2023-07-04      NaN
2023-09-04      NaN
2023-11-10      NaN
2023-11-23      NaN
2024-01-15      NaN
2024-02-19      NaN
2024-05-27      NaN
2024-06-19      NaN
2024-07-04      NaN
2024-09-02      NaN
2024-10-14      NaN
2024-11-11      NaN
2024-11-28      NaN
2025-01-20      NaN
2025-02-17      NaN
2025-05-26      NaN
2025-06-19      NaN
2025-07-04      NaN
2025-09-01      NaN
2025-10-13      NaN
2025-11-11      NaN
2025-11-27      NaN


In [31]:
fx_path = kb_path.parent / "usd_krw_fred.csv"
fred_fx_raw.to_csv(fx_path, index=True)

In [32]:
fx_saved = pd.read_csv(
    fx_path,
    index_col = 0,
    parse_dates = True,
    na_values="."
)

In [34]:
print(fx_saved.shape)
print(fx_saved["DEXKOUS"].isna().sum())
print(fx_saved.loc["2025-12-24":"2025-12-26"])

(1303, 1)
54
                  DEXKOUS
observation_date         
2025-12-24        1443.62
2025-12-25            NaN
2025-12-26        1441.16
